<a href="https://colab.research.google.com/github/alok-dev115/Machine-Learning-Projects/blob/main/fake_new_predictor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [31]:
# import necessary stufs
import pandas as pd
import numpy as np

import re # pattern matching
from nltk.corpus import stopwords # such as 'I', 'our'
from nltk.stem.porter import PorterStemmer # to get root word
from sklearn.feature_extraction.text import TfidfVectorizer # for numerical representation of features
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [32]:
# import stopwword
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [5]:
print(stopwords.words('english'))

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan't", 'she

In [67]:
news_df = pd.read_csv('fake_news_dataset.csv')

In [68]:
news_df.head()

,title,text,date,source,author,category,label
0,Foreign Democrat final.,more tax development both store agreement lawy...,2023-03-10,NY Times,Paula George,Politics,real
1,To offer down resource great point.,probably guess western behind likely next inve...,2022-05-25,Fox News,Joseph Hill,Politics,fake
2,Himself church myself carry.,them identify forward present success risk sev...,2022-09-01,CNN,Julia Robinson,Business,fake
3,You unit its should.,phone which item yard Republican safe where po...,2023-02-07,Reuters,Mr. David Foster DDS,Science,fake
4,Billion believe employee summer how.,wonder myself fact difficult course forget exa...,2023-04-03,CNN,Austin Walker,Technology,fake


In [69]:
news_df['text']

,text
0,more tax development both store agreement lawy...
1,probably guess western behind likely next inve...
2,them identify forward present success risk sev...
3,phone which item yard Republican safe where po...
4,wonder myself fact difficult course forget exa...
...,...
19995,hit and television I change very our happy doo...
19996,fear most meet rock even sea value design stan...
19997,activity loss very provide eye west create wha...
19998,term point general common training watch respo...


In [70]:
news_df.shape

(20000, 7)

In [73]:
news_df['content'] = news_df['author'] + ' ' + news_df['title']

In [74]:
news_df.isnull().sum()

,0
title,0
text,0
date,0
source,1000
author,1000
category,0
label,0
content,1000


In [75]:
news_df = news_df.fillna('')

In [76]:
news_df.isnull().sum()

,0
title,0
text,0
date,0
source,0
author,0
category,0
label,0
content,0


In [77]:
port_stem = PorterStemmer()

In [78]:
# here content is text in the dataframe
def stemming(content):
    stemmed_content = re.sub('[^a-zA-Z]',' ',content)
    stemmed_content = stemmed_content.lower()
    stemmed_content = stemmed_content.split()
    stemmed_content = [port_stem.stem(word) for word in stemmed_content if not word in stopwords.words('english')]
    stemmed_content = ' '.join(stemmed_content)
    return stemmed_content

In [79]:
news_df['content'] = news_df['content'].apply(stemming)

In [81]:
X = news_df['content']
Y = news_df['label'].map({'fake': 0, 'real': 1})

In [82]:
print(X, Y)
Y.value_counts()

0                      paula georg foreign democrat final
1                   joseph hill offer resourc great point
2                             julia robinson church carri
3                                 mr david foster dd unit
4             austin walker billion believ employe summer
                               ...                       
19995                           gari mile hous parti born
19996    maria mcbride though nation peopl mayb price box
19997              kristen franklin yet exist experi unit
19998                         david wise school wide item
19999         jame peterson offer chair cover senior born
Name: content, Length: 20000, dtype: object 0        1
1        0
2        0
3        0
4        0
        ..
19995    0
19996    1
19997    1
19998    0
19999    0
Name: label, Length: 20000, dtype: int64


,count
label,
0,10056
1,9944


In [83]:
# vectorizing
vectorizer = TfidfVectorizer()
vectorizer.fit(X)

X = vectorizer.transform(X)

In [84]:
print(X, Y)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 132014 stored elements and shape (20000, 2320)>
  Coords	Values
  (0, 553)	0.39517271425495315
  (0, 729)	0.3873328871734741
  (0, 760)	0.43250659207690884
  (0, 820)	0.4682904962975366
  (0, 1607)	0.5361471992929359
  (1, 855)	0.4083375673326562
  (1, 945)	0.4329505464318429
  (1, 1106)	0.3921172748965544
  (1, 1544)	0.407233989112382
  (1, 1650)	0.407233989112382
  (1, 1761)	0.4004737874070532
  (2, 340)	0.46335926841900094
  (2, 402)	0.4694627504698104
  (2, 1116)	0.5792618396588832
  (2, 1802)	0.4789140167154591
  (3, 529)	0.38846745035715885
  (3, 535)	0.4677433916557515
  (3, 765)	0.531911235278912
  (3, 1474)	0.3629974574156946
  (3, 2168)	0.46434086969508237
  (4, 128)	0.44019669365176084
  (4, 182)	0.39326674627005814
  (4, 209)	0.3992720864489549
  (4, 655)	0.41543723634042506
  (4, 2054)	0.39538652340301494
  :	:
  (19996, 1316)	0.38074773255956035
  (19996, 1345)	0.3402791515027633
  (19996, 1349)	0.4440574089090

In [85]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size = 0.2, stratify=Y, random_state=2)

In [86]:
model = LogisticRegression()

In [87]:
model.fit(X_train, Y_train)

LogisticRegression()

In [88]:
train_pred = model.predict(X_train)
train_acc = accuracy_score(train_pred, Y_train)
print(train_acc)

0.6411875


In [90]:
test_prediction = model.predict(X_test)
test_data_accuracy = accuracy_score(test_prediction, Y_test)
print(test_data_accuracy)

0.49975
